# AlphaTrain Pillar 2i: Expert V2 (10x More Data)

Same Pillar 2f recipe that worked (val_weight=0.001, ranking + anchor).
10x more expert data: 12.8M states from 5,310 games (mean score 5,255).

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `expert_v2_pairwise.pt.gz` — expert V2 data (880 MB compressed)
3. `pillar2f_best.pt` — base model (already on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model
os.makedirs('/content/alphatrain/data', exist_ok=True)
for f in ['pillar2f_best.pt']:
    src = os.path.join(DRIVE, f)
    dst = os.path.join('/content/alphatrain/data', f)
    if not os.path.exists(dst):
        print(f'Copying {f}...')
        shutil.copy(src, dst)
    print(f'{f}: {os.path.getsize(dst)/1e6:.0f} MB')

# Decompress expert V2 data
gz_src = os.path.join(DRIVE, 'expert_v2_pairwise.pt.gz')
pt_dst = '/content/alphatrain/data/expert_v2_pairwise.pt'
if not os.path.exists(pt_dst):
    print('Decompressing expert_v2_pairwise.pt.gz...')
    t0 = time.time()
    !gzip -dc {gz_src} > {pt_dst}
    print(f'Done in {time.time()-t0:.0f}s: {os.path.getsize(pt_dst)/1e9:.1f} GB')
else:
    print(f'expert_v2_pairwise.pt: {os.path.getsize(pt_dst)/1e9:.1f} GB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

# Verify data
d = torch.load('/content/alphatrain/data/expert_v2_pairwise.pt', weights_only=True, map_location='cpu')
print(f"\nExpert V2: {d['boards'].shape[0]:,} states, {d['n_pairs']:,} pairs")
print(f"  has_pairs={'good_boards' in d}, max_score={d['max_score']}, bins={d['num_value_bins']}")
del d

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2i: Same recipe as Pillar 2f but with 10x more data.
# Expert V2: 12.8M states from 5,310 tournament games (mean 5,255).
# val_weight=0.001 prevents backbone war.
# Warm start from Pillar 2f.
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/expert_v2_pairwise.pt \
    --gpu-data --amp \
    --value-bins 1 --val-weight 0.001 --anchor-weight 0.001 --rank-weight 1.0 \
    --resume alphatrain/data/pillar2f_best.pt --warm-start \
    --epochs 10 --batch-size 8192 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2i_best.pt \
    --save-dir /content/checkpoints/pillar2i

In [ ]:
# Evaluate policy score (Pillar 2f baseline was 315/381)
%cd /content
!python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/pillar2i/best.pt \
    --games 50 --seed 42

In [ ]:
# Copy final model to Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2i/{f}'
    dst = f'{DRIVE}/pillar2i_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')